# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `book/chapters/ch06_eda_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 그래프나 표를 많이 만드는 것이 아니라, 데이터를 보며 **현재 데이터로 답할 수 있는 질문**을 만들고 그 질문을 지표와 코드로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- EDA 결과표는 `reports/` 폴더에 CSV와 Markdown 보고서로 저장합니다.
- EDA 결과는 최종 결론이 아니라, 다음 분석 질문을 만드는 중간 산출물입니다.


## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [21]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: g:\내 드라이브\dev_GD\llm-data-analysis-study\notebooks
프로젝트 루트: g:\내 드라이브\dev_GD\llm-data-analysis-study
전처리 데이터 폴더: g:\내 드라이브\dev_GD\llm-data-analysis-study\data\processed
보고서 폴더: g:\내 드라이브\dev_GD\llm-data-analysis-study\reports


## 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [22]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


전처리 데이터 불러오기 완료


## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [23]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (96, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (752, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [24]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


### 전처리 데이터 확인 결과

Chapter 05에서 생성한 `customers`, `products`, `orders`, `order_items`의 clean CSV 4개를 정상적으로 불러왔으며 각 Dataset의 Shape는 이전 전처리 결과와 일치하였다.

`order_date`를 datetime 형식으로 변환한 결과 `NaT` 3건이 확인되었으며 유효한 주문 날짜의 범위는 2025-07-01부터 2028-12-31까지로 나타났다.

향후 기간별 EDA에서는 날짜 결측과 분석 기간 외 데이터의 영향을 검토할 필요가 있다. 또한 기존 `order_month`와 변환된 `order_date`의 정합성을 확인해야 한다.

**검증 결과: PASS (데이터 로딩 및 구조 확인 완료, 유의사항 유지)**

## 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [25]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'metric': [
        '고객 수, 평균 나이',
        '카테고리별 상품 수',
        '주문 수, 비율',
        '총매출, 매출 비중',
        '월별 매출, 월별 주문 수',
        '고객별 총매출, 주문 횟수',
    ],
    'required_data': [
        'customers',
        'products',
        'orders',
        'order_items, products',
        'orders, order_items',
        'customers, orders, order_items',
    ],
})

questions


,analysis_area,question,metric,required_data
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,"고객 수, 평균 나이",customers
1,상품,어떤 카테고리의 상품이 많은가?,카테고리별 상품 수,products
2,주문,주문 상태와 결제수단 분포는 어떤가?,"주문 수, 비율",orders
3,매출,카테고리별 매출은 어떻게 다른가?,"총매출, 매출 비중","order_items, products"
4,시간,월별 매출과 주문 수는 어떻게 변하는가?,"월별 매출, 월별 주문 수","orders, order_items"
5,고객 가치,구매 금액이 높은 고객은 누구인가?,"고객별 총매출, 주문 횟수","customers, orders, order_items"


## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [26]:
customers['age'].describe()


count    150.000000
mean      43.913333
std       17.802397
min       18.000000
25%       30.000000
50%       41.000000
75%       56.750000
max      150.000000
Name: age, dtype: float64

In [27]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,부산,23,15.33
1,대구,21,14.00
2,광주,21,14.00
3,서울,18,12.00
4,울산,17,11.33
5,인천,16,10.67
6,수원,10,6.67
7,성남,9,6.00
8,고양,8,5.33
9,대전,5,3.33


In [28]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,M,80,53.33
1,F,70,46.67


고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가?
- 고객의 평균 나이는 어느 정도인가?
- 성별 고객 수는 어떻게 분포하는가?
- 도시별 구매 금액도 차이가 있는가?


## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [29]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,패션,16,16.67
1,뷰티,16,16.67
2,식품,16,16.67
3,도서,13,13.54
4,전자기기,13,13.54
5,스포츠,10,10.42
6,생활용품,9,9.38
7,전자 기기,1,1.04
8,생활 용품,1,1.04
9,뷰 티,1,1.04


In [30]:
products['price'].describe()


count    9.600000e+01
mean     2.048958e+05
std      1.011848e+06
min      5.000000e+03
25%      5.275000e+04
50%      9.450000e+04
75%      1.532500e+05
max      9.999999e+06
Name: price, dtype: float64

In [32]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
2,뷰티,16,710937.0,18000.0,9999999.0
3,생활 용품,1,124000.0,124000.0,124000.0
6,식품,16,120812.0,11000.0,195000.0
8,전자기기,13,120231.0,38000.0,178000.0
9,패션,16,115125.0,28000.0,197000.0
0,도서,13,103692.0,34000.0,182000.0
5,스포츠,10,78400.0,5000.0,183000.0
4,생활용품,9,69556.0,29000.0,147000.0
1,뷰 티,1,54000.0,54000.0,54000.0
7,전자 기기,1,21000.0,21000.0,21000.0


## 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [33]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,180,60.0
1,cancelled,63,21.0
2,refunded,57,19.0


In [34]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,90,30.00
1,card,72,24.00
2,bank_transfer,70,23.33
3,naver_pay,68,22.67


In [35]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00
주문 월 개수: 16


관찰: 고객의 평균 age는 43.91세이며, 최댓값은 150세이다. 전체 300건의 주문 중 completed는 180건(60%)으로 나타났다.

해석: 극단적인 age 값은 고객 연령 분포 해석에 영향을 줄 수 있으며 전체 주문과 완료 주문은 구분하여 분석할 필요가 있다.

한계: 150세의 입력 오류 여부는 확정할 수 없으며 주문 상태의 구성만으로 취소·환불 원인을 판단할 수 없다.

## 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [36]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [38]:
sales_items = order_items.merge(
    products,
    on='product_id',
    how='left',
)

print('병합 전 order_items:', order_items.shape)
print('병합 후 sales_items:', sales_items.shape)
print('상품명 누락:', sales_items['product_name'].isna().sum())
print('카테고리 누락:', sales_items['category'].isna().sum())

sales_items.head()


병합 전 order_items: (752, 6)
병합 후 sales_items: (752, 9)
상품명 누락: 26
카테고리 누락: 26


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price
0,1,1,39,2.0,172000.0,344000.0,뷰티 상품 039,뷰티,172000.0
1,2,2,26,4.0,79000.0,316000.0,뷰티 상품 026,뷰티,79000.0
2,3,2,46,3.0,87000.0,261000.0,생활용품 상품 046,생활용품,87000.0
3,4,2,80,1.0,187000.0,187000.0,패션 상품 080,패션,187000.0
4,5,2,91,5.0,125000.0,625000.0,전자기기 상품 091,전자기기,125000.0


## 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [39]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales


,category,total_quantity,total_sales,sales_ratio
9,패션,545.0,64890000.0,26.39
6,식품,394.0,47044000.0,19.13
8,전자기기,303.0,37542000.0,15.27
2,뷰티,282.0,28614000.0,11.64
0,도서,277.0,28246000.0,11.49
5,스포츠,254.0,20651000.0,8.40
4,생활용품,213.0,14303000.0,5.82
3,생활 용품,27.0,3348000.0,1.36
1,뷰 티,19.0,1026000.0,0.42
7,전자 기기,12.0,252000.0,0.10


In [40]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,패션,545.0,64890000.0,26.39,115125.0
1,식품,394.0,47044000.0,19.13,120812.0
2,전자기기,303.0,37542000.0,15.27,120231.0
3,뷰티,282.0,28614000.0,11.64,710937.0
4,도서,277.0,28246000.0,11.49,103692.0
5,스포츠,254.0,20651000.0,8.40,78400.0
6,생활용품,213.0,14303000.0,5.82,69556.0
7,생활 용품,27.0,3348000.0,1.36,124000.0
8,뷰 티,19.0,1026000.0,0.42,54000.0
9,전자 기기,12.0,252000.0,0.10,21000.0


해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [44]:
order_sales = order_items.merge(
    orders,
    on='order_id',
    how='left',
)

order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

print('병합 후 order_sales:', order_sales.shape)
print('order_date 누락:', order_sales['order_date'].isna().sum())

order_sales.head()


병합 후 order_sales: (752, 12)
order_date 누락: 11


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,39,2.0,172000.0,344000.0,144.0,2026-07-12,naver_pay,completed,2026-07,Sunday
1,2,2,26,4.0,79000.0,316000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
2,3,2,46,3.0,87000.0,261000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
3,4,2,80,1.0,187000.0,187000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
4,5,2,91,5.0,125000.0,625000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday


In [45]:
monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales


,order_month,total_sales,order_count,avg_order_value
0,2025-07,13729000.0,19,722579.0
1,2025-08,32639000.0,25,1305560.0
2,2025-09,13058000.0,12,1088167.0
3,2025-10,16680000.0,22,758182.0
4,2025-11,15080000.0,23,655652.0
5,2025-12,16152000.0,23,702261.0
6,2026-01,19053000.0,26,732808.0
7,2026-02,14207000.0,19,747737.0
8,2026-03,13841000.0,22,629136.0
9,2026-04,21007000.0,26,807962.0


월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [47]:
customer_sales_base = order_sales.merge(
    customers,
    on='customer_id',
    how='left',
)

group_columns = ['customer_id', 'city']
if 'name' in customer_sales_base.columns:
    group_columns = ['customer_id', 'name', 'city']

customer_sales = (
    customer_sales_base
    .groupby(group_columns, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)

customer_sales.head(10)


,customer_id,name,city,order_count,total_sales,avg_order_value
15,18.0,서하현,대전,2,16666000.0,8333000.0
2,3.0,이예우,서울,7,8909000.0,1272714.0
17,20.0,김성희,성남,6,6200000.0,1033333.0
45,52.0,박유현,울산,5,5388000.0,1077600.0
11,14.0,임서영,대구,4,5204000.0,1301000.0
19,22.0,강영민,부산,4,5029000.0,1257250.0
48,55.0,장유미,부산,6,4850000.0,808333.0
74,90.0,윤서미,수원,5,4847000.0,969400.0
12,15.0,강수영,광주,5,4281000.0,856200.0
77,93.0,윤도우,울산,4,4001000.0,1000250.0


## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [49]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary


,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [50]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))


[WindowsPath('g:/내 드라이브/dev_GD/llm-data-analysis-study/reports/ch06_customer_city.csv'),
 WindowsPath('g:/내 드라이브/dev_GD/llm-data-analysis-study/reports/ch06_product_category.csv'),
 WindowsPath('g:/내 드라이브/dev_GD/llm-data-analysis-study/reports/ch06_category_sales.csv'),
 WindowsPath('g:/내 드라이브/dev_GD/llm-data-analysis-study/reports/ch06_monthly_sales.csv'),
 WindowsPath('g:/내 드라이브/dev_GD/llm-data-analysis-study/reports/ch06_customer_sales.csv'),
 WindowsPath('g:/내 드라이브/dev_GD/llm-data-analysis-study/reports/ch06_eda_questions.csv')]

## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [52]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, 매출 관점의 기본 현황을 탐색했습니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path)


EDA 요약 보고서 저장 완료: g:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch06_eda_summary.md


## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [61]:
from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()


dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [62]:
module_results['category_sales'].head()


,category,total_quantity,total_sales,sales_ratio
9,패션,545.0,64890000.0,26.39
6,식품,394.0,47044000.0,19.13
8,전자기기,303.0,37542000.0,15.27
2,뷰티,282.0,28614000.0,11.64
0,도서,277.0,28246000.0,11.49


## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


## 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 검증 가능한 EDA 질문 10개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋과 pandas 기능을 함께 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.
```


## 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [ ]:
# 과제 1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.

# 도시별 completed 주문 금액
city_completed_sales = (
    customer_sales_completed_base
    .groupby('city', as_index=False, dropna=False)
    .agg(total_sales=('line_total', 'sum'))
)

# 도시별 고객 수와 구매 금액 연결
city_comparison = customer_city[
    ['city', 'customer_count']
].merge(
    city_completed_sales,
    on='city',
    how='outer',
    validate='one_to_one'
)

# 고객 수 및 금액 결측 처리
city_comparison['customer_count'] = (
    city_comparison['customer_count'].fillna(0).astype(int)
)
city_comparison['total_sales'] = (
    city_comparison['total_sales'].fillna(0)
)

# 구매 금액 기준 내림차순 정렬
city_comparison = city_comparison.sort_values(
    'total_sales', ascending=False
)

print('고객 수 합계:', city_comparison['customer_count'].sum())
print('구매 금액 합계:', city_comparison['total_sales'].sum())

city_comparison


고객 수 합계: 150
구매 금액 합계: 141340000.0


,city,customer_count,total_sales
5,부산,23,24916000.0
6,서울,18,22211000.0
2,광주,21,19866000.0
10,인천,16,15723000.0
3,대구,21,14781000.0
8,수원,10,12220000.0
9,울산,17,11545000.0
7,성남,9,11288000.0
1,고양,8,7222000.0
0,Unknown,2,820000.0


### 과제 1
부산은 고객 수 23명, completed 주문 금액 24,916,000원으로 모두 가장 높았다. 반면 서울은 대구보다 고객 수가 적지만 구매 금액은 더 높게 나타났다.  
이를 통해 고객 수와 구매 금액이 반드시 비례하지는 않음을 확인했으며 구매 금액 총합은 원본과 일치하였다.

In [64]:
# 과제 2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.

category_comparison = product_category[
    ['category', 'product_count']
].merge(
    category_sales_completed[
        ['category', 'total_sales', 'sales_ratio']
    ],
    on='category',
    how='outer',
    validate='one_to_one'
)

category_comparison = category_comparison.sort_values(
    'product_count', ascending=False, na_position='last'
)

print('등록 상품 수 합계:', category_comparison['product_count'].sum())
print('completed 금액 합계:', category_comparison['total_sales'].sum())

category_comparison


등록 상품 수 합계: 96.0
completed 금액 합계: 141340000.0


,category,product_count,total_sales,sales_ratio
6,식품,16.0,25035000.0,17.71
2,뷰티,16.0,17103000.0,12.10
9,패션,16.0,32940000.0,23.31
0,도서,13.0,17663000.0,12.50
8,전자기기,13.0,21485000.0,15.20
5,스포츠,10.0,13237000.0,9.37
4,생활용품,9.0,7001000.0,4.95
1,뷰 티,1.0,648000.0,0.46
3,생활 용품,1.0,2108000.0,1.49
7,전자 기기,1.0,105000.0,0.07


### 과제 2
식품, 뷰티, 패션은 등록 상품 수가 각각 16개로 같지만 `completed` 주문 금액은 패션이 32,940,000원으로 가장 높았다.  
따라서 등록 상품 수가 많다고 반드시 매출이 높은 것은 아니며 판매 수량과 상품 단가 등의 추가 확인이 필요하다.

In [65]:
# 과제 3. 결제수단별 평균 주문 금액을 계산하세요.

payment_sales_completed = (
    completed_order_sales
    .groupby('payment_method', as_index=False, dropna=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
)

payment_sales_completed['avg_order_value'] = (
    payment_sales_completed['total_sales']
    / payment_sales_completed['order_count']
).round(0)

payment_sales_completed = payment_sales_completed.sort_values(
    'avg_order_value', ascending=False
)

print('completed 금액 합계:', payment_sales_completed['total_sales'].sum())
print('completed 주문 수:', payment_sales_completed['order_count'].sum())

payment_sales_completed



completed 금액 합계: 141340000.0
completed 주문 수: 179


,payment_method,order_count,total_sales,avg_order_value
1,card,40,37542000.0,938550.0
0,bank_transfer,49,39461000.0,805327.0
2,kakao_pay,50,36861000.0,737220.0
3,naver_pay,40,27476000.0,686900.0


### 과제 3
`completed` 주문 기준 평균 주문 금액은 card가 938,550원으로 가장 높았고 naver_pay가 686,900원으로 가장 낮았다.  
주문 횟수는 kakao_pay가 50건으로 가장 많았으나 평균 주문 금액은 상대적으로 낮아 결제수단별 주문 빈도와 평균 구매 금액에 차이가 있음을 확인하였다.

In [66]:
# 과제 4. 월별·카테고리별 매출 요약표를 만들어 보세요.

monthly_category_base = completed_order_sales.merge(
    products[['product_id', 'category']],
    on='product_id',
    how='left',
    validate='many_to_one'
)

monthly_category_sales = (
    monthly_category_base
    .groupby(['order_month', 'category'], as_index=False, dropna=False)
    .agg(total_sales=('line_total', 'sum'))
    .sort_values(
        ['order_month', 'total_sales'],
        ascending=[True, False],
        na_position='last'
    )
)

print('병합 전 상세 행:', len(completed_order_sales))
print('병합 후 상세 행:', len(monthly_category_base))
print('월별·카테고리별 합계:', monthly_category_sales['total_sales'].sum())

monthly_category_sales.head(20)


병합 전 상세 행: 446
병합 후 상세 행: 446
월별·카테고리별 합계: 141340000.0


,order_month,category,total_sales
6,2025-07,패션,3163000.0
4,2025-07,식품,2123000.0
5,2025-07,전자기기,1319000.0
1,2025-07,뷰티,1133000.0
0,2025-07,도서,1040000.0
2,2025-07,생활용품,406000.0
3,2025-07,스포츠,295000.0
7,2025-07,NaN,82000.0
13,2025-08,전자기기,2547000.0
14,2025-08,패션,2246000.0


### 과제 4
`completed` 주문을 월별·카테고리별로 집계한 결과 2025년 7월에는 패션이 3,163,000원, 8월에는 전자기기가 2,547,000원으로 해당 월에서 가장 높은 금액을 기록하였다.  
월별 주요 카테고리에 차이가 있으며 전체 집계 합계는 141,340,000원으로 원본과 일치하였다.

In [ ]:

# 과제 5. 반복 구매 고객과 고액 단발 구매 고객 구분 

customer_check = customer_sales_completed[
    customer_sales_completed['city'].notna()
].copy()

high_value_cutoff = customer_check['avg_order_value'].quantile(0.75)

repeat_customers = customer_check[
    customer_check['order_count'] >= 2
]

high_single_customers = customer_check[
    (customer_check['order_count'] == 1) &
    (customer_check['avg_order_value'] >= high_value_cutoff)
]

print('고액 기준:', high_value_cutoff)
print('반복 구매 고객 수:', len(repeat_customers))
print('고액 단발 구매 고객 수:', len(high_single_customers))

high_single_customers[
    ['customer_id', 'order_count', 'avg_order_value']
]


고액 기준: 1026500.0
반복 구매 고객 수: 53
고액 단발 구매 고객 수: 11


,customer_id,order_count,avg_order_value
33,46.0,1,2300000.0
34,47.0,1,1472000.0
53,81.0,1,1416000.0
9,17.0,1,1406000.0
96,149.0,1,1389000.0
46,64.0,1,1368000.0
15,24.0,1,1103000.0
28,40.0,1,1100000.0
5,11.0,1,1096000.0
10,19.0,1,1084000.0


### 과제 5
`completed` 주문 2건 이상을 반복 구매 고객으로, 주문 1건이면서 평균 주문 금액이 상위 25%(1,026,500원 이상)에 해당하면 고액 단발 구매 고객으로 정의하였다. 분석 결과 반복 구매 고객은 53명, 고액 단발 구매 고객은 11명으로 확인되었다.

### 과제 6. LLM을 활용한 분석 질문 수정

온라인 쇼핑몰의 `customers`, `products`, `orders`, `order_items` 데이터를 분석하고 있습니다.

다음 질문을 현재 데이터로 검증 가능한 EDA 질문으로 수정해 주세요.

기존 질문: "고객 불만이 많아서 환불이 발생했는가?"

현재 보유한 데이터 구조

customers: customer_id, gender, age, city

orders: order_id, customer_id, order_date, payment_method, order_status

order_items: order_id, product_id, quantity, unit_price, line_total

products: product_id, category, price

order_status에는 completed, cancelled, refunded가 존재하지만, 고객 불만이나 환불 사유에 대한 Column은 없습니다.

다음 조건을 지켜 주세요.

1. 현재 데이터에서 확인 가능한 내용만 활용할 것
2. 원인을 추측하거나 인과관계를 단정하지 않을 것
3. 수정한 질문과 필요한 Dataset 및 Column을 명시할 것
4. 사용할 지표와 검증 방법을 제안할 것
5. 현재 데이터로 판단할 수 없는 한계를 설명할 것
6. Python 코드 사용하지 말것

수정된 EDA 질문은 1개만 제안해 주세요.

LLM 답변:  
고객 불만이 환불의 원인인지 확인하는 질문을 LLM에 제시한 결과 "상품 카테고리(category)별로 전체 주문 건수 대비 환불(refunded) 발생 비율에 유의미한 차이가 존재하는가?"를 제안받았다.  
다만 분석 기준이 모호하여 '카테고리별 해당 카테고리가 포함된 주문 중 refunded 주문 비율은 어떻게 다른가?'로 수정하였다. 이를 통해 현재 데이터로 검증 가능한 질문으로 구체화하였다.

## 나의 추가 EDA

### 선택한 질문

상품 카테고리별로 해당 카테고리가 포함된 주문 중 `refunded` 상태의 주문 비율은 어떻게 다른가?

### 선택 이유

LLM이 제안한 카테고리별 환불 비율 분석을 통해 환불이 상대적으로 많이 발생하는 상품군을 파악하고 향후 상품 품질 및 고객 만족도 개선이 필요한 영역을 탐색하고자 한다.

### 분석 범위

`completed`, `cancelled`, `refunded` 상태의 주문을 대상으로 한다. 하나의 주문에 동일한 카테고리의 상품이 여러 개 포함되어도 해당 카테고리의 주문은 1건으로 계산한다.

### 필요한 데이터와 컬럼

- `orders`: `order_id`, `order_status`
- `order_items`: `order_id`, `product_id`
- `products`: `product_id`, `category`

### 사용할 지표

- `category_order_count`: 카테고리가 포함된 전체 주문 수
- `refunded_order_count`: 해당 카테고리가 포함된 `refunded` 주문 수
- `refund_ratio`: `refunded_order_count / category_order_count × 100`

### 검증 방법

`order_id`, `product_id` 병합 전후 행 수와 미매칭을 확인하고 동일 주문·카테고리의 중복 집계를 방지한다.  
상품 정보가 누락된 데이터는 별도로 보존하며 카테고리별 주문 수와 환불 비율을 비교한다.

In [68]:

# 나의 추가 EDA: 카테고리별 refunded 주문 비율

# 1. 주문 상태 및 상품 카테고리 연결
refund_base = (
    order_items[['order_id', 'product_id']]
    .merge(
        orders[['order_id', 'order_status']],
        on='order_id',
        how='left',
        validate='many_to_one'
    )
    .merge(
        products[['product_id', 'category']],
        on='product_id',
        how='left',
        validate='many_to_one'
    )
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(refund_base))
print('주문 상태 누락 행:', refund_base['order_status'].isna().sum())
print('카테고리 누락 행:', refund_base['category'].isna().sum())

# 2. 주문 상태 확인 및 중복 제거
refund_orders = refund_base[
    refund_base['order_status'].isin(
        ['completed', 'cancelled', 'refunded']
    )
].drop_duplicates(['order_id', 'category'])

# 3. 카테고리별 주문 수와 환불 주문 수
refund_category = (
    refund_orders
    .groupby('category', as_index=False, dropna=False)
    .agg(
        category_order_count=('order_id', 'nunique'),
        refunded_order_count=(
            'order_status',
            lambda x: (x == 'refunded').sum()
        )
    )
)

# 4. 환불 비율 계산
refund_category['refund_ratio'] = (
    refund_category['refunded_order_count']
    / refund_category['category_order_count'] * 100
).round(2)

refund_category = refund_category.sort_values(
    'refund_ratio', ascending=False
)

refund_category


병합 전 행 수: 752
병합 후 행 수: 752
주문 상태 누락 행: 2
카테고리 누락 행: 26


,category,category_order_count,refunded_order_count,refund_ratio
7,전자 기기,4,2,50.00
1,뷰 티,7,2,28.57
10,NaN,26,7,26.92
4,생활용품,67,16,23.88
0,도서,83,19,22.89
6,식품,114,23,20.18
8,전자기기,89,16,17.98
2,뷰티,88,15,17.05
9,패션,108,18,16.67
5,스포츠,73,11,15.07


### 결과 관찰

`전자 기기`의 `refunded` 주문 비율은 50.00%로 가장 높았으나 전체 주문이 4건에 불과하였다. 상대적으로 주문 수가 많은 `생활용품`과 `도서`에서는 각각 23.88%, 22.89%의 비율이 확인되었다.

### 나의 해석과 판단

단순 환불 비율뿐 아니라 주문 수를 함께 고려할 필요가 있다. 특히 `생활용품`과 `도서`는 환불이 발생한 주문의 특성을 추가로 분석할 가치가 있다고 판단하였다.

### 현재 분석의 한계

주문 단위의 `refunded` 상태만으로 특정 상품의 실제 환불 여부나 환불 사유를 판단할 수 없다.

### 다음 질문
refunded 주문을 경험한 고객 중 이후 completed 주문으로 재구매한 비율은 얼마나 되며 카테고리별로 차이가 있는가?

# Chapter 06 답안 양식. 데이터를 보며 질문을 만드는 EDA

> 이 내용을 `chapter06.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름:김희철
- GitHub ID:KHeeCheol
- 작성일:2026-10-19
- 최종 제출 URL:https://github.com/KHeeCheol/llm-data-analysis-study/blob/main/notebooks/chapter06.ipynb

## 1. 내가 정한 EDA 질문

| 질문 | 분석 범위 | 지표 | 검증 방법 |
|---|---|---|---|
| 카테고리별 매출은 어떻게 다른가? | `orders`, `order_items`, `products`를 활용하여 `completed` 주문의 카테고리별 금액 분석 | `total_sales`, `sales_ratio` | `order_id`, `product_id` 병합 및 미매칭 건수를 확인하고, 카테고리별 금액 합계를 `completed` 주문 원본 합계와 비교 |
| 월별 매출과 주문 수는 어떻게 변하는가? | `orders`, `order_items`를 활용하여 `completed` 주문의 월별 금액 및 주문 건수 분석 | `total_sales`, `order_count`, `avg_order_value` | 월별 금액 합계와 원본 합계를 비교하고, `order_date` 결측 및 분석 기간에 따른 영향을 확인 |
| 결제수단별 `completed` 주문의 평균 주문 금액과 주문 횟수는 어떻게 다른가? | `orders`, `order_items`를 활용하여 `payment_method`별 완료 주문의 구매 특성 분석 | `order_count`, `total_sales`, `avg_order_value` | 고유 `order_id`를 기준으로 주문 수를 계산하고, 결제수단별 금액 합계와 원본 합계 및 평균 주문 금액의 계산 결과를 비교 |

### 왜 이 질문을 선택했는가?

카테고리별 매출과 월별 매출 추이를 통해 상품군별 금액 기여도와 시기별 구매 양상을 파악하고자 하였다.   
추가로 결제수단별 주문 횟수와 평균 주문 금액을 비교하여 결제방식에 따른 구매 행동의 차이가 존재하는지 확인하고자 한다.  
세 질문 모두 현재 보유한 데이터로 분석이 가능하며, 금액 집계는 `completed` 주문을 기준으로 수행한다.  
또한 Chapter 05에서 확인된 참조 불일치, 날짜 결측 및 이상값이 집계 결과에 미칠 수 있는 영향을 검토하여 분석 결과의 신뢰성을 확보하고자 한다.

## 2. 핵심 집계 결과
- completed 원본 합계:141,340,000원
- 카테고리별 합계:141,340,000원
- 월별 합계:141,340,000원
- 총합 일치 여부: 일치

![핵심 EDA 결과](images/step02_eda.png)

### 결과 관찰
completed 주문 기준 카테고리별 금액은 패션이 32,940,000원(23.31%)으로 가장 높았으며 식품과 전자기기가 뒤를 이었다.   
월별 금액은 2026년 8월이 16,980,000원으로 가장 높았다. 카테고리별·월별 금액 합계는 원본 합계와 일치하였다.
### 나의 해석과 판단
카테고리 및 월별 금액에 차이가 나타났으나 이를 상품 선호도나 특정 시기의 수요 증가로 단정하기는 어렵다.  
판매 수량, 주문 수, 평균 주문 금액 및 상품 단가를 함께 고려해야 한다.
### 업무·분석적 의미
카테고리별 금액 비중과 월별 주문 특성을 비교하면 주요 상품군과 구매 규모가 집중된 시기를 파악하는 기초 자료로 활용할 수 있다.  
또한 동일한 completed 주문 범위에서 원본 금액과 그룹별 합계를 비교하여 집계 과정의 누락 여부를 검증할 수 있었다.
### 한계와 추가 확인 사항
completed 주문 상세 중 category가 누락된 16개 행(4,015,000원)과 order_date가 누락된 6개 행(1,903,000원)이 확인되었다.  
이들은 총합 검증을 위해 별도 그룹에 포함했으나 실제 카테고리 및 월별 분포 해석에는 한계가 있다.  
또한 카테고리 표기 불일치 극단값 및 비정상적인 날짜의 영향을 추가로 확인할 필요가 있다.

## 3. 관찰 → 가설 → 추가 검증

### 사례 1. 주력 상품 중심의 판매 전략
- 관찰: 패션·식품·뷰티는 등록 상품 수가 각각 16개로 같지만 `completed` 주문 금액은 패션이 32,940,000원으로 가장 높았다.
- 가설: 패션의 높은 구매 금액은 전체 상품에 고르게 분포하기보다 소수 인기 상품에 구매가 집중된 결과일 수 있다.
- 추가 검증: 카테고리별 상위 3개 상품의 금액 비중과 실제 판매된 상품 수를 비교하여 주력 상품 집중 광고와 카테고리 전체 프로모션 중 적합한 전략을 검토한다.

### 사례 2. 지역별 고객 마케팅 전략
- 관찰: 서울은 등록 고객 18명, 구매 금액 22,211,000원으로 대구(21명, 14,781,000원)보다 고객 수가 적지만 구매 금액은 높았다.
- 가설: 서울의 높은 구매 금액은 구매 고객의 비율이나 고객당 반복 주문 횟수가 상대적으로 높기 때문일 수 있다.
- 추가 검증: 도시별 실제 구매 고객 비율, 고객당 주문 횟수 및 평균 구매 금액을 비교하여 지역별 신규 고객 확보와 재구매 유도 전략의 우선순위를 검토한다.

### 사례 3. 결제수단별 고액 구매 특성
- 관찰: `completed` 주문 기준 card의 평균 주문 금액은 938,550원으로 가장 높았으나 주문 횟수는 kakao_pay가 50건으로 가장 많았다.
- 가설: card 결제 주문에는 고액 주문의 비중이 상대적으로 높아 평균 주문 금액이 크게 나타났을 가능성이 있다.
- 추가 검증: 결제수단별 주문 금액 중앙값, 고액 주문 비율 및 구매 카테고리 구성을 비교하여 고액 구매 고객을 대상으로 한 결제 혜택의 활용 가능성을 검토한다.

## 4. LLM 질문 확장
- Safe Context: customers, products, orders, order_items의 컬럼 구조와 order_status 정보를 제공하고, 환불 사유 데이터가 없음을 명시하였다.
- Prompt: "고객 불만이 많아서 환불이 발생했는가?"라는 질문을 현재 데이터로 검증 가능한 EDA 질문으로 수정하도록 요청하였다.
- LLM 제안 요약: 카테고리별 전체 주문 대비 refunded 발생 비율에 유의미한 차이가 있는지 분석하는 질문을 제안하였다.
- 실제 사용 여부: 수정 후 사용
- 판단 이유: 현재 데이터로 분석 가능하지만 카테고리별 주문 수의 정의와 '유의미한 차이'의 판단 기준이 불명확하였다.
- 사람이 수정한 내용: "상품 카테고리별로 해당 카테고리가 포함된 주문 중 refunded 상태의 주문 비율은 어떻게 다른가?"로 수정하여 분석 범위를 구체화하였다.

![LLM EDA 질문 확장](images/step04_llm.png)

## 5. 다음 분석 우선순위

1. 카테고리별 주력 상품 매출 집중도 분석: 소수 상품에 구매가 집중되는지 확인하여 집중 마케팅과 카테고리 전체 프로모션 전략 비교
2. 지역별 구매 고객 특성 분석: 구매 고객 비율, 반복 주문 및 평균 구매 금액을 비교하여 지역별 고객 확보와 유지 전략 검토
3. 결제수단별 고액 구매 특성 분석: 주문 금액 분포와 상품 구성을 비교하여 고액 구매 고객 대상 결제 혜택 전략 검토

### 가장 먼저 확인하고 싶은 이유

세 분석 모두 상품, 지역, 결제수단별 구매 특성을 구체화하여 마케팅 자원의 효율적인 배분에 활용할 수 있다.  
그중 카테고리별 매출 차이는 이미 확인되었으며 상품별 구매 금액과 판매 집중도를 추가로 분석하면 주력 상품 선정 및 프로모션 전략에 직접적인 근거를 제공할 수 있어 우선순위가 가장 높다고 판단하였다.

### 현재 데이터만으로 단정할 수 없는 것
구매 금액이나 주문 횟수의 차이만으로 실제 고객 선호도와 구매 원인을 판단하기 어렵다.  
또한 광고비, 상품별 이익률, 프로모션 효과 등의 정보가 없어 제안한 마케팅 전략의 실질적인 수익성이나 효과를 단정할 수 없다.

### 이번 장에서 EDA에 대해 새롭게 이해한 점
EDA는 단순히 데이터를 집계하는 과정이 아니라 분석 질문에 맞는 범위를 정의하고 결과의 신뢰성을 검증하는 과정임을 이해하였다.  
특히 동일한 데이터라도 `completed` 필터와 결측값 처리 기준에 따라 분석 결과가 달라지므로 수치의 정확성과 해석의 타당성을 함께 확인하는 것이 중요하다고 판단하였다.

## 최종 체크
- [O] 질문과 지표가 연결되어 있습니다.
- [O] 집계 총합을 검증했습니다.
- [O] 관찰과 가설을 구분했습니다.
- [O] 다음 검증 질문을 작성했습니다.
- [O] LLM 제안을 검증했습니다.
- [O] 최종 Notebook URL을 제출합니다.

## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.
